# 02 — RQ2: AI vs matched-control adoption rates

RQ2 compares the **AI cohort** (n=2,803) against a **1:1 matched
control cohort** (n=1,744) on whether each repo has each
tool/category configured at the cutoff date.

Per family (5 categories, 4 retained tools after sparse-band
exclusion), the analyst computed:

- Fisher exact OR with Haldane–Anscombe 0.5 continuity for empty
  cells, plus 95% CI from log-OR ± 1.96·SE.
- Adjusted GLM (Binomial logit) controlling for `log_stars`,
  `language`, `repo_age`, `activity`, `owner_type`, with cluster-
  robust HC1 SE on `owner` (3,750 clusters).
- BH FDR correction within each family (categories: m=5; tools: m=4
  after dropping 15 sparse-band tools).

All numbers below come from `analysis/tables/rq2_*.csv` and
`rq2_headline.json`.

## Provenance

In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

REPO_ROOT = (
    Path(__file__).resolve().parents[2]
    if "__file__" in globals()
    else Path.cwd().parents[1]
)
TABLES = REPO_ROOT / "analysis" / "tables"
FIGURES = REPO_ROOT / "analysis" / "figures"


class MissingArtifact(FileNotFoundError):
    pass


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise MissingArtifact(f"missing canonical CSV: {path}")
    return pd.read_csv(path)


def load_json(path: Path) -> dict:
    if not path.exists():
        raise MissingArtifact(f"missing canonical JSON: {path}")
    return json.loads(path.read_text())


prov = load_json(TABLES / "rq2_provenance.json")
print(f"run_id            : {prov['run_id']}")
print(f"aidev_dataset_sha : {prov['aidev_dataset_sha']}")
print(f"compute_script    : {prov['compute_script']}")
print(f"generated_at_utc  : {prov['generated_at_utc']}")
print()
print("BH family definition:")
print(
    f"  alpha                                  : {prov['bh_family_definition']['alpha']}"
)
print(
    f"  bh_method                              : {prov['bh_family_definition']['bh_method']}"
)
print(
    f"  bh_family_size_categories              : {prov['bh_family_definition']['bh_family_size_categories']}"
)
print(
    f"  bh_family_size_tools_after_exclusion   : {prov['bh_family_definition']['bh_family_size_tools_after_exclusion']}"
)
print(
    f"  sparse_band_threshold                  : {prov['bh_family_definition']['sparse_band_threshold']}"
)
print(
    f"  sparse_band_threshold_basis            : {prov['bh_family_definition']['sparse_band_threshold_basis']}"
)
print()
print("configs hashes:")
for name, info in prov["configs_hashes"].items():
    print(f"  {name:32s}  {info['version_header']}  sha256={info['sha256'][:16]}…")

run_id            : 2026-04-25
aidev_dataset_sha : v3
compute_script    : analysis/scripts/rq2_compute.py
generated_at_utc  : 2026-05-01T18:23:02+00:00

BH family definition:
  alpha                                  : 0.05
  bh_method                              : fdr_bh
  bh_family_size_categories              : 5
  bh_family_size_tools_after_exclusion   : 4
  sparse_band_threshold                  : 0.01
  sparse_band_threshold_basis            : control adoption rate < 1% in repo_security_adoption_wide.parquet

configs hashes:
  agent_fingerprints.yaml           version: 2026-04-25  sha256=5c5a03502c2e9121…
  security_bots.txt                 version: 2026-04-26  sha256=edbd29526cce9e97…
  security_patterns.yaml            version: 2026-04-26  sha256=12c66325bfafe6bb…
  tools.yaml                        version: 2026-04-25  sha256=6d2df0bc3bba6117…


## Sparse-band exclusion (read this before the forest plot)

15 tools have a control-arm adoption rate < 1%, so the BH family
would be dominated by Haldane-shifted CIs that span orders of
magnitude. The analyst excludes these from the BH family and reports
them descriptively only. They appear in the per-tool table below
**without** an adjusted p_value.

Two RQ2 categories — `secrets` and `fuzzing` — are pre-declared
underpowered (achieved power 0.36 and 0.20 respectively at OR=1.8 / 2.0
vs the locked α=0.05; see `power_analysis.csv`). They appear in the
headline table but their negative findings should be read as
**not-informative**, not as evidence of "no effect".

In [2]:
headline = load_json(TABLES / "rq2_headline.json")
print(f"sparse-band excluded tools (n={headline['n_sparse_band_tools_excluded']}):")
for t in headline["sparse_band_excluded"]:
    print(f"  - {t}")
print()
print(
    f"underpowered families (pre-declared): n = {headline['n_underpowered_families']} (secrets, fuzzing)"
)

sparse-band excluded tools (n=15):
  - anchore
  - bandit
  - checkov
  - claude_code_security_review
  - gitleaks
  - govulncheck
  - harden_runner
  - microsoft_security_devops
  - oss_fuzz
  - semgrep
  - snyk
  - sonarqube
  - tfsec
  - trivy
  - trufflehog

underpowered families (pre-declared): n = 2 (secrets, fuzzing)


## Headline: BH-significant categories and tools (AI > control)

In [3]:
print("BH-significant categories where AI > control:")
pd.DataFrame(headline["bh_sig_categories"])

BH-significant categories where AI > control:


,OR,OR_hi,OR_lo,level,p_adj
0,3.1166,3.8663,2.5123,sast,6.921553e-29
1,3.3640,3.9360,2.8750,sca,6.266300e-58
2,2.8658,4.5664,1.7985,ci_hardening,2.610663e-06


In [4]:
print("BH-significant tools where AI > control (after sparse-band exclusion):")
pd.DataFrame(headline["bh_sig_tools"])

BH-significant tools where AI > control (after sparse-band exclusion):


,OR,OR_hi,OR_lo,level,p_adj
0,3.1286,3.9018,2.5086,codeql,7.175655e-28
1,3.1225,3.6861,2.6451,dependabot,4.089718e-46
2,2.5651,4.2000,1.5666,ossf_scorecard,7.018494e-05
3,3.8603,5.9220,2.5164,renovate,3.436627e-12


## RQ2 main table

`OR` is Fisher exact (or Haldane-shifted in empty cells; tools where
Fisher could not be computed have a blank `OR_fisher`). `adj_OR` is
the cluster-robust adjusted logit OR; `p_adj` is BH-corrected
**within family** (categories or tools). Rows in the sparse-band have
`bh_family_member=False` and are reported descriptively only.

In [5]:
rq2 = load_csv(TABLES / "rq2_main.csv")
rq2

,family,level,n_AI,n_ctrl,ai_rate,ctrl_rate,OR,OR_lo,OR_hi,OR_fisher,...,p_adj,bh_family_member,adj_OR,adj_OR_lo,adj_OR_hi,adj_p,adj_cov_type,adj_n_clusters,smd_summary,underpowered_flag
0,category,sast,2803,1744,0.174813,0.063647,3.1166,2.5123,3.8663,3.1166,...,6.921553e-29,True,0.4579,0.3529,0.5943,4.256507e-09,cluster,3750,0.3482,False
1,category,sca,2803,1744,0.348198,0.137041,3.3640,2.8750,3.9360,3.3640,...,6.266300e-58,True,0.4293,0.3603,0.5115,3.061251e-21,cluster,3750,0.5082,False
2,category,secrets,2803,1744,0.009633,0.005161,1.8750,0.8797,3.9963,1.8750,...,1.513390e-01,True,0.8048,NaN,NaN,NaN,cluster,3750,0.0522,True
3,category,fuzzing,2803,1744,0.003924,0.001720,2.2864,0.6370,8.2070,2.2864,...,2.728075e-01,True,0.7331,NaN,NaN,NaN,cluster,3750,0.0416,True
4,category,ci_hardening,2803,1744,0.035319,0.012615,2.8658,1.7985,4.5664,2.8658,...,2.610663e-06,True,0.4988,0.3110,0.8000,3.901948e-03,cluster,3750,0.1489,False
5,tool,anchore,2803,1744,0.006778,0.003440,1.9769,0.7880,4.9596,1.9769,...,NaN,False,0.6082,0.2262,1.6350,3.243518e-01,cluster,3750,0.0468,False
6,tool,bandit,2803,1744,0.000714,0.001147,0.6219,0.0875,4.4191,0.6219,...,NaN,False,2.4517,NaN,NaN,NaN,cluster,3750,-0.0142,False
7,tool,checkov,2803,1744,0.000357,0.001720,0.2071,0.0215,1.9927,0.2071,...,NaN,False,7.1947,NaN,NaN,NaN,cluster,3750,-0.0423,False
8,tool,claude_code_security_review,2803,1744,0.000714,0.000000,3.1135,0.1494,64.8906,NaN,...,NaN,False,0.0000,NaN,NaN,NaN,cluster,3750,0.0378,False
9,tool,codeql,2803,1744,0.166964,0.060206,3.1286,2.5086,3.9018,3.1286,...,7.175655e-28,True,0.4562,0.3486,0.5971,1.102456e-08,cluster,3750,0.3413,False


![RQ2 forest plot, BH family members](../figures/rq2_tool_forest.png)

![RQ2 category configured rates](../figures/rq2_category_rates.png)

## RQ2 robustness battery

30 robustness rows across pre-declared variants (per README §8):

- `v1_strict_relaxed_fingerprints` — deferred, requires Phase B rerun
  (would mutate the control cohort).
- `v4_lang_strat` — re-run RQ2 within each top-7 language stratum.
- other variants per the analyst's robustness.py.

In [6]:
robust = load_csv(TABLES / "rq2_robustness.csv")
print(f"total RQ2 robustness rows: {len(robust)}")
robust["variant"].value_counts()

total RQ2 robustness rows: 30


variant
v4_lang_strat                     25
v1_strict_relaxed_fingerprints     5
Name: count, dtype: int64

### Try changing X

Filter `robust` to one variant at a time, e.g.
`robust.query("variant == 'v4_lang_strat' and outcome == 'sast'")`,
to inspect the per-language SAST robustness.

In [7]:
robust.query("variant == 'v4_lang_strat' and outcome == 'sast'")[
    [
        "stratum",
        "n_AI",
        "n_ctrl",
        "ai_rate",
        "ctrl_rate",
        "or_or_irr",
        "ci95_lo",
        "ci95_hi",
        "p_raw",
    ]
]

,stratum,n_AI,n_ctrl,ai_rate,ctrl_rate,or_or_irr,ci95_lo,ci95_hi,p_raw
5,TypeScript,647.0,343.0,0.160742,0.064140,2.794576,1.728894,4.517138,8.352396e-06
10,Python,529.0,261.0,0.166352,0.068966,2.693878,1.584604,4.579679,9.466713e-05
15,Go,242.0,169.0,0.301653,0.165680,2.175190,1.332997,3.549484,1.649700e-03
20,C#,220.0,156.0,0.231818,0.044872,6.423500,2.828742,14.586464,2.129666e-07
25,JavaScript,190.0,113.0,0.094737,0.035398,2.851744,0.940133,8.650314,6.687890e-02
